In [7]:
pip install -U datasets


ERROR: Operation cancelled by user
^C


In [3]:
!pip install --upgrade datasets fsspec


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 13.8 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.2
    Uninstalling fsspec-2025.3.2:
      Successfully uninstalled fsspec-2025.3.2
  Attempting uninstall: datasets
    Found existing installation: datasets 2.14.4
    Uninstalling datasets-2.14.4:
      Successfully uninstalled datasets-2.14.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torch 2.6.0+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cublas-cu12 12.5.3.2 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cuda-cupti-cu12==12.4.127; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cuda-cupti-cu12 

In [9]:
import re
import requests
from datasets import load_dataset

# ─────────────────────────────────────────────────────────────────────────
# 1) 데이터셋 로드
# ─────────────────────────────────────────────────────────────────────────
ds = load_dataset("2tle/korean-curse-filtering-dataset")
split_name = list(ds.keys())[0]  # 보통 "test"
print("Available split:", split_name)

# ─────────────────────────────────────────────────────────────────────────
# 2) 기존 룰베이스 사전(bad_words) 로드
# ─────────────────────────────────────────────────────────────────────────
def load_bad_words(url):
    r = requests.get(url)
    r.raise_for_status()
    return r.json().get("words", [])

with open('fword_list.txt', 'r', encoding='utf-8') as f:
    bad_words = [ line.strip().lower() for line in f if line.strip() ]
print(f"원래 bad_words 사전 길이: {len(bad_words)}")

# ─────────────────────────────────────────────────────────────────────────
# 3) ds[split_name]["text"]에서 '|' 이후 텍스트 추출 + 쉼표 기준으로 분리
# ─────────────────────────────────────────────────────────────────────────
all_curse_tokens = set()

for example in ds[split_name]:
    text = example["text"]  # 예: "…|욕1,욕2,욕3"
    # 3-1) '|' 기호가 있는 경우, 그 뒤쪽만 슬라이싱
    if "|" in text:
        after_bar = text.split("|", 1)[1]  # '|' 뒤에 나오는 모든 문자열
        # 3-2) 쉼표(,)로 분리 → 각각 strip() 해서 add
        for tok in after_bar.split(","):
            tok = tok.strip().lower()
            if tok:
                all_curse_tokens.add(tok)

print(f"데이터셋에서 추출된 curse 토큰(고유) 수: {len(all_curse_tokens)}")

# ─────────────────────────────────────────────────────────────────────────
# 4) 기존 bad_words에 없는 단어만 추가
# ─────────────────────────────────────────────────────────────────────────
initial_count = len(bad_words)
for tok in all_curse_tokens:
    # 이미 bad_words가 소문자/공백제거 형태이므로 그대로 비교
    if tok not in bad_words:
        bad_words.append(tok)

added_count = len(bad_words) - initial_count
print(f"기존 사전 크기: {initial_count}")
print(f"새로 추가된 단어 개수: {added_count}")
print(f"최종 룰베이스 사전 크기: {len(bad_words)}")

# ─────────────────────────────────────────────────────────────────────────
# 5) 확장된 룰베이스로 간단히 테스트
# ─────────────────────────────────────────────────────────────────────────
def contains_bad_word_loose(text, bad_words):
    found = set()
    lowered = text.lower()
    for bw in bad_words:
        if bw and (bw in lowered):
            found.add(bw)
    return found

tests = [
    "너 꺼져버려?",
    "멍청하지 너",
    "이건 좀 맘에 안드네요!"
]

for t in tests:
    bad_found = contains_bad_word_loose(t, bad_words)
    print(f"문장: {t}")
    print(f"검출된 욕설 단어: {bad_found}")
    print("-" * 40)


Available split: test
원래 bad_words 사전 길이: 973
데이터셋에서 추출된 curse 토큰(고유) 수: 232
기존 사전 크기: 973
새로 추가된 단어 개수: 188
최종 룰베이스 사전 크기: 1161

--- 룰베이스 테스트 예시 ---
문장: 너 꺼져버려?
검출된 욕설 단어: {'꺼져'}
----------------------------------------
문장: 멍청하지 너
검출된 욕설 단어: {'멍청'}
----------------------------------------
문장: 이건 좀 맘에 안드네요!
검출된 욕설 단어: set()
----------------------------------------


In [10]:
# ───────────────────────────────────────────────────────────────────────────────
# 6) “확장된 룰베이스” 생성 (기존 + new_tokens) 및 파일로 저장
# ───────────────────────────────────────────────────────────────────────────────

output_path = "extended_bad_words.txt"
with open(output_path, "w", encoding="utf-8") as fout:
    for w in bad_words:
        fout.write(w + "\n")

print(f"▶ extended_bad_words.txt 파일이 생성되었습니다: {output_path}")

▶ extended_bad_words.txt 파일이 생성되었습니다: extended_bad_words.txt
